In [1]:
import pandas as pd

df = pd.read_parquet("hf://datasets/dair-ai/emotion/unsplit/train-00000-of-00001.parquet")

In [6]:
df.head()

,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1
3,i was feeling a little low few days back,0
4,i beleive that i am much more sensitive to oth...,2


In [8]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())
print(df.head(3).to_string(index=False))
print("\nlabel values and counts:")
print(df["label"].value_counts().sort_index())

shape: (416809, 2)
columns: ['text', 'label']
                                                                                                                                                    text  label
                                        i feel awful about it too because it s my job to get him in a position to succeed and it just didn t happen here      0
                                                                                                                                   im alone i feel awful      0
ive probably mentioned this before but i really do feel proud of myself for actually keeping up with my new years resolution of monthly and weekly goals      1

label values and counts:
label
0    121187
1    141067
2     34554
3     57317
4     47712
5     14972
Name: count, dtype: int64


In [2]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

c:\Users\shash\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)



transformers: 4.52.3
torch: 2.7.0+cpu
cuda available: False


In [3]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
MODEL_NAME = "bert-base-uncased"
MAX_ROWS = 12_000  # Set to None to train and evaluate on the full dataset.
MAX_LENGTH = 128
LABEL_NAMES = ["sadness", "joy", "love", "anger", "fear", "surprise"]
NUM_LABELS = len(LABEL_NAMES)

# The Hugging Face dair-ai/emotion labels use these IDs.
label_to_id = {name: index for index, name in enumerate(LABEL_NAMES)}
id_to_label = {index: name for name, index in label_to_id.items()}

model_data = df[["text", "label"]].dropna().copy()
model_data["text"] = model_data["text"].astype(str)
model_data["label"] = model_data["label"].astype(int)
if not set(model_data["label"].unique()).issubset(set(range(NUM_LABELS))):
    raise ValueError("Expected label IDs from 0 to 5.")

if MAX_ROWS is not None and len(model_data) > MAX_ROWS:
    model_data, _ = train_test_split(
        model_data,
        train_size=MAX_ROWS,
        random_state=SEED,
        stratify=model_data["label"],
    )

train_data, temp_data = train_test_split(
    model_data,
    test_size=0.2,
    random_state=SEED,
    stratify=model_data["label"],
)
valid_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_data["label"],
)

print(f"Train: {len(train_data):,} | Validation: {len(valid_data):,} | Test: {len(test_data):,}")
print("Training label counts:\n", train_data["label"].value_counts().sort_index())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length):
        self.labels = frame["label"].to_numpy(dtype=np.int64)
        self.encodings = tokenizer(
            frame["text"].tolist(),
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: torch.tensor(value[index]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

train_dataset = EmotionDataset(train_data, tokenizer, MAX_LENGTH)
valid_dataset = EmotionDataset(valid_data, tokenizer, MAX_LENGTH)
test_dataset = EmotionDataset(test_data, tokenizer, MAX_LENGTH)
print("Tokenized example length:", len(train_dataset[0]["input_ids"]))

Train: 9,600 | Validation: 1,200 | Test: 1,200
Training label counts:
 label
0    2791
1    3249
2     796
3    1320
4    1099
5     345
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\shash\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shash\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenized example length: 128


In [4]:
from transformers import Trainer, TrainingArguments


def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
    }


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id_to_label,
    label2id=label_to_id,
)

training_args = TrainingArguments(
    output_dir="./emotion-bert-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)
print("Trainer ready on:", training_args.device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainer ready on: cpu


In [5]:
def train_and_evaluate():
    train_result = trainer.train()
    trainer.save_model("./emotion-bert")
    tokenizer.save_pretrained("./emotion-bert")

    test_output = trainer.predict(test_dataset, metric_key_prefix="test")
    predictions = np.argmax(test_output.predictions, axis=-1)
    test_labels = test_output.label_ids

    print("Test metrics:")
    print(pd.Series(test_output.metrics).to_string())
    print("\nPer-class report:")
    print(
        classification_report(
            test_labels,
            predictions,
            labels=list(range(NUM_LABELS)),
            target_names=LABEL_NAMES,
            digits=4,
            zero_division=0,
        )
    )

    ConfusionMatrixDisplay.from_predictions(
        test_labels,
        predictions,
        labels=list(range(NUM_LABELS)),
        display_labels=LABEL_NAMES,
        xticks_rotation="vertical",
        cmap="Blues",
    )
    return {"train_result": train_result, "test_metrics": test_output.metrics}

# Run train_and_evaluate() in the next cell to start fine-tuning and test evaluation.

In [ ]:
training_results = train_and_evaluate()

c:\Users\shash\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [6]:
def predict_emotion(text, top_k=3):
    model_inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )
    model_inputs = {key: value.to(trainer.model.device) for key, value in model_inputs.items()}

    trainer.model.eval()
    with torch.inference_mode():
        probabilities = torch.softmax(trainer.model(**model_inputs).logits[0], dim=-1)

    top_indices = torch.topk(probabilities, k=min(top_k, NUM_LABELS)).indices.tolist()
    return [
        {"emotion": id_to_label[index], "probability": float(probabilities[index])}
        for index in top_indices
    ]

# Example after training: predict_emotion("I finally got the job I wanted!")

In [7]:
predictions = predict_emotion("I finally got the job I wanted!")
print("Predictions:", predictions)

Predictions: [{'emotion': 'fear', 'probability': 0.22952821850776672}, {'emotion': 'sadness', 'probability': 0.20067188143730164}, {'emotion': 'anger', 'probability': 0.14910241961479187}]
